# KPI Dictionary & Data Quality Profile — Retail Orders

**Purpose:** profile the supplied intentionally imperfect retail orders dataset and test a practical data-quality contract.

**Decision owner:** Head of Sales (business), with Finance as revenue sign-off and Data Engineering as pipeline owner.

## How to run
1. Download `retail-orders-raw.csv` and `retail-data-dictionary.csv` from the official links in the assignment.
2. Place both CSV files in the same folder as this notebook.
3. Install dependencies if needed: `pip install pandas numpy`.
4. Run all cells from top to bottom.

The notebook also tries the official URLs if local files are not present. It adapts to common column-name variants and prints which fields were found. A KPI is marked unavailable if its required source fields are absent; no missing data is fabricated.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timezone

ORDERS_URL = "https://rabtechacademy.in/starter-files/data/retail-orders-raw.csv"
DICT_URL = "https://rabtechacademy.in/starter-files/data/retail-data-dictionary.csv"

def load_csv(local_name, url):
    path = Path(local_name)
    if path.exists():
        return pd.read_csv(path)
    try:
        return pd.read_csv(url)
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not find {local_name} locally and could not download it. "
            f"Download it from the assignment link and place it beside this notebook. "
            f"Original error: {exc}"
        )

df_raw = load_csv("retail-orders-raw.csv", ORDERS_URL)
try:
    dictionary = load_csv("retail-data-dictionary.csv", DICT_URL)
except FileNotFoundError:
    dictionary = pd.DataFrame()

print("Profile run (UTC):", datetime.now(timezone.utc).isoformat())
print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("Columns:", list(df_raw.columns))
if not dictionary.empty:
    display(dictionary.head(20))


In [ ]:
# Normalize column names for robust matching, while retaining the original dataframe.
def norm(name):
    return "".join(ch for ch in str(name).lower().strip() if ch.isalnum())

column_lookup = {norm(c): c for c in df_raw.columns}

def find_col(*candidates):
    for candidate in candidates:
        if norm(candidate) in column_lookup:
            return column_lookup[norm(candidate)]
    # fuzzy fallback: candidate text contained in normalized column name
    for candidate in candidates:
        target = norm(candidate)
        for normalized, original in column_lookup.items():
            if target and (target in normalized or normalized in target):
                return original
    return None

COL = {
    "order_id": find_col("order_id", "orderid", "transaction_id", "transactionid", "order_number"),
    "line_id": find_col("order_line_id", "line_id", "order_item_id", "line_number", "item_id"),
    "customer": find_col("customer_id", "customerid", "client_id"),
    "date": find_col("order_date", "orderdate", "transaction_date", "date", "created_at"),
    "status": find_col("status", "order_status", "payment_status"),
    "revenue": find_col("total_spent", "total_amount", "order_amount", "amount", "revenue", "sales", "line_total"),
    "unit_price": find_col("price_per_unit", "unit_price", "price"),
    "quantity": find_col("quantity", "qty", "units"),
    "discount": find_col("discount_applied", "discount", "is_discounted"),
    "category": find_col("category", "product_category"),
    "channel": find_col("location", "channel", "sales_channel"),
    "payment": find_col("payment_method", "paymentmethod"),
    "promised_date": find_col("promised_delivery_date", "expected_delivery_date"),
    "delivered_date": find_col("delivered_date", "actual_delivery_date")
}
print("Detected fields:")
for key, value in COL.items():
    print(f"{key:16} -> {value}")


In [ ]:
# Basic structural profile
profile = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "non_null_count": [int(df_raw[c].notna().sum()) for c in df_raw.columns],
    "missing_count": [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    "missing_pct": [round(float(df_raw[c].isna().mean() * 100), 2) for c in df_raw.columns],
    "distinct_count": [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns],
    "sample_values": [", ".join(map(str, df_raw[c].dropna().astype(str).unique()[:5])) for c in df_raw.columns]
})
display(profile)
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))


In [ ]:
# Standardize a working copy and safely parse common data types.
df = df_raw.copy()
for key in ["date", "promised_date", "delivered_date"]:
    col = COL[key]
    if col:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False)

for key in ["revenue", "unit_price", "quantity"]:
    col = COL[key]
    if col:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False).str.replace("₹", "", regex=False).str.strip(), errors="coerce")

# Normalize text fields without overwriting source raw file.
for key in ["order_id", "customer", "status", "discount", "category", "channel", "payment"]:
    col = COL[key]
    if col:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})
print("Working data prepared. Raw input remains unchanged.")


In [ ]:
# Executable data-quality checks: completeness, uniqueness, validity, consistency, freshness.
results = []

def add_check(dimension, check, bad_count, denominator, threshold_pct, severity, details=""):
    denominator = int(denominator)
    bad_count = int(bad_count)
    pct = (100 * bad_count / denominator) if denominator else 0.0
    passed = pct <= threshold_pct
    results.append({
        "dimension": dimension, "check": check, "bad_rows": bad_count,
        "rows_tested": denominator, "bad_pct": round(pct, 3),
        "threshold_pct": threshold_pct, "passed": passed,
        "severity": severity, "details": details
    })

n = len(df)
required_keys = [("order_id", 0.5), ("date", 0.5), ("status", 0.5), ("revenue", 1.0), ("quantity", 1.0)]
for key, threshold in required_keys:
    col = COL[key]
    if col:
        missing = df[col].isna() | (df[col].astype("string").str.strip() == "")
        add_check("Completeness", f"{key} populated", missing.sum(), n, threshold, "High", f"Source column: {col}")
    else:
        results.append({"dimension":"Completeness","check":f"{key} field exists","bad_rows":n,"rows_tested":n,"bad_pct":100.0,"threshold_pct":0.0,"passed":False,"severity":"High","details":"Required field not detected; map the correct column or document as unavailable."})

# Uniqueness: exact duplicate rows are always reported; business key only tested if a line identifier exists.
add_check("Uniqueness", "Exact duplicate rows", df.duplicated().sum(), n, 0.0, "Medium", "Exact duplicates are candidates for review, not automatic deletion.")
if COL["order_id"] and COL["line_id"]:
    dup_key = df.duplicated(subset=[COL["order_id"], COL["line_id"]], keep=False)
    add_check("Uniqueness", "Order ID + line ID duplicate key", dup_key.sum(), n, 0.0, "High", "Validate grain before deduplication.")
elif COL["order_id"]:
    results.append({"dimension":"Uniqueness","check":"Order ID uniqueness","bad_rows":int(df[COL["order_id"]].duplicated().sum()),"rows_tested":n,"bad_pct":round(100*df[COL["order_id"]].duplicated().sum()/n,3) if n else 0,"threshold_pct":None,"passed":None,"severity":"Info","details":"Order IDs can repeat in order-line data; do not treat as duplicate unless the dictionary says one row per order."})

# Validity checks
if COL["date"]:
    bad = df[COL["date"]].isna().sum()
    add_check("Validity", "Date parses", bad, n, 0.5, "High")
if COL["revenue"]:
    amount = df[COL["revenue"]]
    add_check("Validity", "Revenue numeric and non-negative", (amount.isna() | (amount < 0)).sum(), n, 0.5, "High")
if COL["quantity"]:
    qty = df[COL["quantity"]]
    add_check("Validity", "Quantity numeric and positive", (qty.isna() | (qty <= 0)).sum(), n, 0.5, "High")
if COL["discount"]:
    disc = df[COL["discount"]].astype("string").str.lower().str.strip()
    allowed = {"true","false","1","0","yes","no","y","n","1.0","0.0"}
    invalid = disc.notna() & ~disc.isin(allowed)
    add_check("Validity", "Discount flag in approved boolean domain", invalid.sum(), disc.notna().sum(), 0.5, "Medium", "Review allowed values against dictionary.")
if COL["status"]:
    statuses = sorted(df[COL["status"]].dropna().astype(str).str.lower().unique().tolist())
    print("Observed status values (approve mapping before KPI use):", statuses)

# Consistency: line revenue should match unit price × quantity where both exist.
if COL["unit_price"] and COL["quantity"] and COL["revenue"]:
    expected = df[COL["unit_price"]] * df[COL["quantity"]]
    actual = df[COL["revenue"]]
    applicable = expected.notna() & actual.notna()
    mismatch = applicable & ~np.isclose(expected, actual, rtol=0.0, atol=0.02)
    add_check("Consistency", "Revenue = unit price × quantity", mismatch.sum(), applicable.sum(), 0.5, "High", "Tolerance ±0.02 currency units; adjust if documented discounts/tax apply.")
else:
    results.append({"dimension":"Consistency","check":"Revenue arithmetic reconciliation","bad_rows":0,"rows_tested":0,"bad_pct":0.0,"threshold_pct":0.5,"passed":None,"severity":"Info","details":"Skipped: unit price, quantity, or revenue column not detected."})

# Freshness is only testable when a trustworthy timestamp exists. Event date is a proxy, not ingestion time.
if COL["date"] and df[COL["date"]].notna().any():
    latest = df[COL["date"]].max()
    print("Latest event date:", latest)
    print("Freshness note: this is event-time freshness, not ingestion freshness. Compare with source SLA before treating it as a pipeline failure.")
else:
    print("Freshness check unavailable: no parseable event timestamp detected.")

quality_results = pd.DataFrame(results)
display(quality_results)


In [ ]:
# KPI calculations with conservative status handling.
# Status values are printed for review. Adjust these sets to match the official dictionary before business publication.
COMPLETED_VALUES = {"completed", "complete", "paid", "fulfilled", "delivered", "success", "successful"}
CANCELLED_VALUES = {"cancelled", "canceled"}
FAILED_VALUES = {"failed", "payment failed", "rejected"}
REFUNDED_VALUES = {"refunded", "returned"}

def eligible_mask(frame):
    status_col = COL["status"]
    if not status_col:
        # No status field means the KPI is a provisional all-row estimate; disclose this limitation.
        return pd.Series(True, index=frame.index)
    values = frame[status_col].astype("string").str.lower().str.strip()
    if values.isin(COMPLETED_VALUES).any():
        return values.isin(COMPLETED_VALUES)
    # Do not assume unknown status semantics; return no eligible rows until mapping is approved.
    return pd.Series(False, index=frame.index)

mask = eligible_mask(df)
eligible = df.loc[mask].copy()
print(f"Eligible rows: {len(eligible)} / {len(df)}")
if COL["status"] and not mask.any():
    print("No statuses matched the default completed set. Update COMPLETED_VALUES using the observed status list and dictionary before KPI publication.")

def distinct_count(frame, col):
    return int(frame[col].nunique(dropna=True)) if col else None

kpi_output = []
def emit(name, value, definition, note=""):
    kpi_output.append({"KPI": name, "Value": value, "Formula / interpretation": definition, "Status / limitation": note})

rev_col, order_col, cust_col, qty_col = COL["revenue"], COL["order_id"], COL["customer"], COL["quantity"]
revenue_total = float(eligible[rev_col].dropna().sum()) if rev_col and len(eligible) else np.nan
orders_total = distinct_count(eligible, order_col) if order_col else None
units_total = float(eligible[qty_col].dropna().sum()) if qty_col and len(eligible) else np.nan
customers_total = distinct_count(eligible, cust_col) if cust_col else None

emit("Total Revenue", revenue_total, "Sum of eligible revenue", "Provisional until status mapping and currency confirmed." if not COL["status"] else "")
emit("Completed Orders", orders_total, "Distinct eligible order IDs", "Unavailable: order ID not detected." if not order_col else "")
emit("Average Order Value", revenue_total / orders_total if orders_total else np.nan, "Eligible revenue / distinct eligible orders", "Requires both revenue and order ID.")
emit("Units Sold", units_total, "Sum of eligible positive quantity", "Check quantity validity results.")
emit("Average Selling Price per Unit", revenue_total / units_total if units_total and units_total > 0 else np.nan, "Eligible revenue / eligible units", "")
emit("Unique Customers", customers_total, "Distinct non-null customer IDs", "Unavailable: customer ID not detected." if not cust_col else "")
emit("Revenue per Customer", revenue_total / customers_total if customers_total else np.nan, "Eligible revenue / distinct customers", "")
if COL["discount"]:
    disc = eligible[COL["discount"]].astype("string").str.lower().str.strip()
    true_values = {"true","1","yes","y","1.0"}
    known_values = true_values | {"false","0","no","n","0.0"}
    known = disc.isin(known_values)
    share = 100 * disc[known].isin(true_values).sum() / known.sum() if known.sum() else np.nan
    emit("Discounted Order/Row Share", share, "Rows with true discount flag / rows with recognized flag × 100", "Row-based unless distinct order aggregation is possible; verify grain.")
else:
    emit("Discounted Order Share", np.nan, "Distinct discounted orders / eligible orders × 100", "Unavailable: discount flag not detected.")
if COL["status"]:
    all_status = df[COL["status"]].astype("string").str.lower().str.strip()
    known_status = all_status.notna()
    cancel_rate = 100 * all_status[known_status].isin(CANCELLED_VALUES).sum() / known_status.sum() if known_status.sum() else np.nan
    emit("Cancellation Rate", cancel_rate, "Cancelled orders/records / records with known status × 100", "Row-based unless distinct order aggregation is possible.")
else:
    emit("Cancellation Rate", np.nan, "Cancelled orders / all orders × 100", "Unavailable: status field not detected.")
if COL["promised_date"] and COL["delivered_date"] and order_col:
    fulfilled = eligible.dropna(subset=[COL["promised_date"], COL["delivered_date"]])
    on_time = fulfilled[COL["delivered_date"]] <= fulfilled[COL["promised_date"]]
    emit("On-time Fulfilment Rate", 100 * on_time.mean() if len(fulfilled) else np.nan, "Fulfilled rows delivered on/before promised date / rows with both dates × 100", "")
else:
    emit("On-time Fulfilment Rate", np.nan, "On-time fulfilled orders / fulfilled orders with valid dates × 100", "Unavailable: promised/actual delivery dates not detected.")
kpis = pd.DataFrame(kpi_output)
display(kpis)


In [ ]:
# Save evidence artifacts beside the notebook.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
profile.to_csv(out_dir / "column_profile.csv", index=False)
quality_results.to_csv(out_dir / "data_quality_results.csv", index=False)
kpis.to_csv(out_dir / "kpi_results.csv", index=False)
print("Saved:")
for p in sorted(out_dir.glob("*.csv")):
    print(" -", p)
